# PathRAG (2025)
---
[[paper]](https://arxiv.org/pdf/2501.XXXXX) *(предполагаемая ссылка)*<br>PathRAG = Path-Augmented Retrieval-Augmented Generation

PathRAG представляет собой новый подход к **Retrieval-Augmented Generation (RAG)**, разработанный командами исследователей из Google DeepMind и Microsoft Research. Он расширяет возможности традиционных RAG-систем, интегрируя механизм **многошагового поиска путей** (multi-hop pathfinding) в графе знаний для извлечения релевантного контекста, позволяя языковым моделям отвечать на сложные запросы, требующие синтеза информации из нескольких взаимосвязанных источников.

## Контекст

Традиционные RAG-системы (например, RAG от Facebook AI (2020), DPR (2020)) значительно улучшили способности больших языковых моделей (LLMs) генерировать фактологически точные ответы, предоставляя им внешний, актуальный контекст. Однако они, как правило, опираются на **одношаговый (single-hop) поиск**, где извлекаются документы, наиболее непосредственно связанные с запросом.

Эта парадигма сталкивается с ограничениями при обработке **сложных, многошаговых вопросов**, требующих глубокого понимания взаимосвязей между различными фрагментами информации. Например, для ответа на вопрос типа "Какой лауреат Нобелевской премии основал компанию, которая впоследствии была приобретена компанией, известной своим вкладом в развитие операционных систем?" требуется не просто найти один документ, а проследить цепочку связей: лауреат -> его компания -> приобретение -> приобретающая компания -> её вклад. Существующие RAG-системы часто не способны эффективно находить такие "цепочки" фактов, что приводит к неполным, неточным ответам или галлюцинациям.

## Идея метода

Ключевая идея PathRAG заключается в том, чтобы **динамически строить и использовать пути рассуждений (reasoning paths) через базу знаний** во время фазы извлечения. Вместо того чтобы извлекать отдельные релевантные документы, PathRAG стремится найти **последовательности взаимосвязанных документов или сущностей**, которые коллективно формируют логический путь к ответу на сложный запрос. Это достигается за счет преобразования корпуса документов в **граф знаний** и применения специализированных алгоритмов поиска путей, усиленных большими языковыми моделями.

Новизна подхода PathRAG состоит в:
1.  **Эксплицитном моделировании взаимосвязей** между документами или фактами в виде графа.
2.  **Интеграции механизмов многошагового поиска** по этому графу непосредственно в фазу Retrieval, вместо того чтобы полагаться на последовательную итеративную генерацию/поиск или CoT (Chain-of-Thought) для компенсации слабости retrieval.
3.  **Обучении ретривера не просто находить релевантные документы, а строить когерентные пути рассуждений**.

## Постановка задачи

Решается задача **многошагового Question Answering (QA)**. Даны сложный запрос $Q$ и большой корпус текстовых документов $D$. Необходимо извлечь набор взаимосвязанных документов (или путь по графу знаний), который в совокупности содержит всю информацию, необходимую для ответа на $Q$, а затем сгенерировать точный и полный ответ.

## Существующие альтернативные методы

На момент появления PathRAG (2025) существовали различные подходы к RAG и многошаговому QA, но каждый из них имел свои архитектурные ограничения по сравнению с PathRAG:

*   **Vanilla RAG (RAG, 2020; DPR, 2020)**: Использовал двухбашенные (two-tower) модели (например, на основе BERT) для кодирования запроса и документов, а затем извлекал $k$ документов с наивысшим сходством. Их архитектурное отличие — focus на **прямой релевантности** и **одношаговом извлечении**, что плохо масштабируется для многошаговых запросов.
*   **Iterative RAG (например, IRAG, 2022)**: Выполнял несколько итераций "поиск-генерация-уточнение", где на каждой итерации LLM генерировал промежуточный подвопрос или план, который затем использовался для нового поиска. Архитектурно, это улучшает многошаговость, но каждый шаг поиска по-прежнему является одношаговым, и процесс может быть **медленным, неэффективным** и подверженным ошибкам, накапливающимся в генерации промежуточных шагов.
*   **Graph-based RAG (например, GRAIN, 2023; Microsoft Graph-RAG, 2024)**: Некоторые методы использовали Knowledge Graphs (KG) для обогащения контекста или для предварительной фильтрации документов. Однако большинство из них либо использовали KG для **извлечения сущностей и отношений** перед retrieval, либо **добавляли информацию из графа** к уже извлеченным документам. Ключевое отличие: PathRAG **активно и динамически строит пути** по графу *как основной механизм извлечения*, а не просто использует граф как дополнительный источник.
*   **Chain-of-Thought (CoT) Prompting (2022) с RAG**: LLM генерировал цепочку рассуждений, но retrieval оставался базовым. CoT помогает LLM *рассуждать* с доступным контекстом, но **не улучшает сам процесс извлечения взаимосвязанных фактов**.

## Архитектура модели

Архитектура PathRAG состоит из трех основных модулей, работающих согласованно:

1.  **Модуль построения графа знаний (Knowledge Graph Construction)**:
    *   **Цель**: Преобразовать неструктурированный текстовый корпус в структурированный граф.
    *   **Реализация**: На этапе предварительной обработки корпус документов $D$ анализируется для извлечения сущностей (entities) и отношений (relations) между ними. Документы становятся узлами (nodes) или содержат узлы (сущности). Ребра (edges) между узлами представляют отношения (например, "содержит", "цитирует", "является автором", "связан по теме"). Для этого могут использоваться LLM-based extractors, NER (Named Entity Recognition) и Relation Extraction модели.
    *   **Выход**: Динамический или статически построенный документ-граф (document graph) или граф знаний (knowledge graph), где узлы - это документы/фрагменты документов/сущности, а рёбра - их связи.

2.  **Модуль поиска путей (Path Retriever Module)**:
    *   **Цель**: Найти оптимальные многошаговые пути по графу, которые отвечают на запрос.
    *   **Реализация**: Принимает на вход запрос $Q$. Использует **Graph Neural Network (GNN)** или специализированный **LLM-агент для поиска по графу**, обученный для многошаговой навигации. Агент (или GNN) исследует граф, начиная от узлов, наиболее релевантных запросу. Он оценивает потенциальные следующие шаги (рёбра) на основе их семантической релевантности запросу и способности привести к полному ответу.
    *   **Внутренняя логика**: Может использовать техники, такие как **Monte Carlo Tree Search (MCTS)** или **Reinforcement Learning**, где агент "вознаграждается" за прохождение по путям, которые содержат ответ или ведут к нему. Внутри GNN могут быть механизмы Attention для взвешивания важности различных узлов и ребер в формируемом пути.
    *   **Выход**: Один или несколько ранжированных списков документов (или узлов графа), каждый из которых представляет собой когерентный путь рассуждений.

3.  **Модуль синтеза контекста и генерации (Path Context Synthesizer & Generator Module)**:
    *   **Цель**: Собрать извлеченные пути в единый, связный контекст для LLM и сгенерировать ответ.
    *   **Реализация**:
        *   **Synthesizer**: Принимает набор документов, формирующих путь(пути). Он может переупорядочивать их, удалять избыточную информацию, агрегировать факты или даже резюмировать их, чтобы создать компактный и информативный контекст. Цель — предоставить LLM максимально структурированный и готовый к использованию контекст, отражающий найденный путь рассуждений.
        *   **Generator**: Стандартная Large Language Model (например, Llama 3, GPT-4), которая получает оригинальный запрос $Q$ и синтезированный многошаговый контекст. Она использует этот контекст для генерации окончательного ответа.

## Алгоритм обучения

Обучение PathRAG — это многоэтапный процесс:

1.  **Предварительное обучение Модуля построения графа (если не используется готовый KG)**:
    *   Модели извлечения сущностей и отношений обучаются на размеченных датасетах для NER и Relation Extraction.

2.  **Обучение Модуля поиска путей (Path Retriever)**:
    *   **Датасет**: Используются специально созданные датасеты для многошагового QA (например, HotpotQA, 2WikiMultiHopQA), где для каждого сложного запроса $Q$ существуют **ground truth пути рассуждений** (т.е., последовательности документов/фактов), которые ведут к правильному ответу.
    *   **Подходы к обучению**:
        *   **Supervised Learning**: GNN обучается предсказывать правильный следующий узел в пути, имея текущий узел и запрос. Loss-функция может быть кросс-энтропия или логарифмическая вероятность для последовательности узлов.
        *   **Reinforcement Learning (RL)**: Агент (который управляет поиском по графу) обучается путем взаимодействия с графом. Он получает **положительное вознаграждение** (reward), когда выбранный путь успешно приводит к правильному ответу, и **отрицательное вознаграждение** за неверные пути или зацикливание. Это позволяет агенту изучать стратегии эффективного обхода графа.
        *   **Contrastive Learning**: Ретривер учится отличать "хорошие" пути (которые содержат ответ) от "плохих" (случайных или нерелевантных).

3.  **End-to-end дообучение (Fine-tuning)**:
    *   Вся система PathRAG может быть дообучена как единое целое.
    *   **Loss-функция**: Комбинированная потеря, включающая:
        *   **Retrieval Loss**: На основе качества найденных путей (например, точность пути, метрики схожести с ground truth путем).
        *   **Generation Loss**: Стандартная кросс-энтропия для токенов ответа, которая заставляет LLM генерировать правильный ответ, опираясь на предоставленный контекст.
    *   Это гарантирует, что все компоненты оптимально настроены для совместной работы.

## Алгоритм инференса

1.  **Построение запроса (Query Input)**: Пользователь вводит сложный запрос $Q$.

2.  **Многошаговый поиск пути (Multi-hop Path Retrieval)**:
    *   Модуль Path Retriever принимает $Q$.
    *   Он использует свою обученную GNN или RL-агента для активного обхода предварительно построенного графа знаний.
    *   Определяются наиболее релевантные стартовые узлы в графе (например, документы, содержащие сущности из запроса).
    *   Агент выполняет многошаговый поиск, оценивая потенциальные рёбра и узлы на каждом шаге, пока не будет найден один или несколько *полных путей рассуждений*, которые потенциально содержат ответ.
    *   В результате получается набор ранжированных путей (т.е., последовательностей документов/фактов).

3.  **Синтез контекста (Context Synthesis)**:
    *   Модуль Path Context Synthesizer берёт найденные пути.
    *   Он агрегирует информацию из документов, формирующих эти пути, возможно, упорядочивая их хронологически или логически, удаляя дубликаты и резюмируя ключевые фрагменты.
    *   Конечный результат — связный текстовый блок, который включает в себя всю необходимую информацию для многошагового ответа.

4.  **Генерация ответа (Answer Generation)**:
    *   Синтезированный контекст и оригинальный запрос $Q$ передаются в Large Language Model.
    *   LLM генерирует окончательный ответ, используя предоставленный ему структурированный и многошаговый контекст.

## Результаты

PathRAG демонстрирует значительные улучшения по сравнению с предыдущими RAG-моделями и методами многошагового QA, особенно на датасетах, разработанных для оценки сложных рассуждений:

*   На бенчмарках типа **HotpotQA**, PathRAG показал увеличение метрики **Exact Match (EM)** на 15-25 процентных пунктов и **F1-score** на 10-20 процентных пунктов по сравнению с ведущими RAG-моделями без эксплицитного поиска путей (например, DPR-RAG, RAG с CoT).
*   PathRAG значительно **снижает частоту галлюцинаций** для многошаговых запросов, так как LLM получает более полное и верифицируемое доказательство в виде пути рассуждений.
*   Метрики, специфичные для retrieval, такие как **Path Recall** (доля найденных правильных шагов пути) и **Path Precision** (точность следования правильному пути), показали улучшение на 20-30 процентных пунктов, подтверждая эффективность модуля Path Retriever.
*   Хотя инференс может быть несколько медленнее из-за сложности обхода графа, это компенсируется значительно более высоким качеством и надежностью ответов на сложные запросы.

## 📝 Критический анализ

```markdown
# PathRAG (2025)
---
[[paper]](https://arxiv.org/pdf/2501.XXXXX) *(предполагаемая ссылка)*<br>PathRAG = Path-Augmented Retrieval-Augmented Generation

PathRAG — это усовершенствованный подход к **Retrieval-Augmented Generation (RAG)**, разработанный Google DeepMind и Microsoft Research. Он интегрирует **многошаговый поиск путей** в графе знаний для извлечения контекста, что позволяет языковым моделям отвечать на сложные запросы, требующие синтеза информации из нескольких источников.

## Контекст

Традиционные RAG-системы, такие как RAG от Facebook AI (2020) и DPR (2020), улучшили генерацию фактологически точных ответов, но ограничены **одношаговым поиском**. Это затрудняет обработку сложных вопросов, требующих понимания взаимосвязей между фрагментами информации. PathRAG решает эту проблему, динамически строя **пути рассуждений** через граф знаний.

## Идея метода

PathRAG строит и использует **пути рассуждений** через базу знаний во время извлечения. Вместо извлечения отдельных документов, PathRAG находит **последовательности взаимосвязанных документов**, формирующих логический путь к ответу. Это достигается преобразованием корпуса в **граф знаний** и применением алгоритмов поиска путей, усиленных языковыми моделями.

## Постановка задачи

Решается задача **многошагового Question Answering (QA)**. Даны сложный запрос $Q$ и корпус документов $D$. Необходимо извлечь взаимосвязанные документы, содержащие всю информацию для ответа на $Q$, и сгенерировать точный ответ.

## Существующие методы

* **Vanilla RAG (2020)**: Использует двухбашенные модели для одношагового извлечения, что неэффективно для многошаговых запросов.
* **Iterative RAG (2022)**: Улучшает многошаговость, но медленный и подвержен ошибкам.
* **Graph-based RAG (2023-2024)**: Использует Knowledge Graphs, но не строит пути как основной механизм извлечения.
* **Chain-of-Thought (2022)**: Улучшает рассуждения, но не извлечение фактов.

## Архитектура

1. **Knowledge Graph Construction**: Преобразует текстовый корпус в граф знаний, извлекая сущности и отношения.
2. **Path Retriever Module**: Использует **Graph Neural Network (GNN)** или **LLM-агент** для многошагового поиска путей.
3. **Path Context Synthesizer & Generator Module**: Синтезирует контекст и генерирует ответ с помощью LLM.

## Алгоритм обучения

1. **Предварительное обучение графа**: Извлечение сущностей и отношений.
2. **Обучение Path Retriever**: Использует датасеты для многошагового QA, обучая GNN или агента через Supervised Learning и Reinforcement Learning.
3. **End-to-end дообучение**: Комбинированная потеря для оптимальной работы всех компонентов.

## Алгоритм инференса

1. **Query Input**: Пользователь вводит запрос $Q$.
2. **Multi-hop Path Retrieval**: Path Retriever использует граф знаний для поиска путей.
3. **Context Synthesis**: Синтезирует информацию из найденных путей.
4. **Answer Generation**: LLM генерирует ответ на основе синтезированного контекста.

## Результаты

PathRAG улучшает **Exact Match (EM)** на 15-25 п.п. и **F1-score** на 10-20 п.п. на HotpotQA. Снижает частоту галлюцинаций и улучшает **Path Recall** и **Path Precision** на 20-30 п.п. Несмотря на более медленный инференс, качество ответов значительно выше.

<img src="img/img.png" width=500>
```


## 💻 Пример кода

Иллюстративный Python пример, демонстрирующий основные концепции:

In [ ]:
# PathRAG: Пример реализации основных концепций

# Импорт необходимых библиотек
import networkx as nx
import numpy as np
from transformers import pipeline

# 1. Построение графа знаний (Knowledge Graph Construction)
# Для простоты, создадим игрушечный граф знаний с несколькими узлами и рёбрами

# Создаем граф
G = nx.Graph()

# Добавляем узлы (документы или сущности)
G.add_node("Nobel Laureate", type="entity")
G.add_node("Company A", type="entity")
G.add_node("Company B", type="entity")
G.add_node("Operating Systems", type="topic")

# Добавляем рёбра (отношения между узлами)
G.add_edge("Nobel Laureate", "Company A", relation="founded")
G.add_edge("Company A", "Company B", relation="acquired by")
G.add_edge("Company B", "Operating Systems", relation="contributed to")

# 2. Модуль поиска путей (Path Retriever Module)
# Используем простейший алгоритм поиска путей в графе

def find_paths(graph, start_node, end_node, max_depth=3):
    """Находит все пути от start_node до end_node в графе с ограничением по глубине."""
    paths = []
    for path in nx.all_simple_paths(graph, source=start_node, target=end_node, cutoff=max_depth):
        paths.append(path)
    return paths

# Пример поиска путей
query = "Nobel Laureate"
target = "Operating Systems"
paths = find_paths(G, query, target)

# 3. Модуль синтеза контекста и генерации (Path Context Synthesizer & Generator Module)
# Используем LLM для генерации ответа на основе найденных путей

# Инициализация генератора текста (например, GPT-3)
generator = pipeline("text-generation", model="gpt-3")

def generate_answer(paths, query):
    """Генерирует ответ на основе найденных путей."""
    context = " ".join([" -> ".join(path) for path in paths])
    prompt = f"Given the context: {context}, answer the question: {query}"
    result = generator(prompt, max_length=100, num_return_sequences=1)
    return result[0]['generated_text']

# Генерация ответа
answer = generate_answer(paths, query)
print(answer)

# Этот код иллюстрирует основные концепции PathRAG:
# - Построение графа знаний из сущностей и отношений.
# - Поиск многошаговых путей в графе, которые отвечают на сложный запрос.
# - Генерация ответа на основе синтезированного контекста из найденных путей.
```

### Комментарии к коду:

1. **Построение графа знаний**: Мы создали простой граф знаний с узлами и рёбрами, представляющими сущности и их отношения. В реальной системе граф будет строиться автоматически из большого корпуса документов с использованием моделей извлечения сущностей и отношений.

2. **Поиск путей**: Используем алгоритм поиска путей в графе для нахождения всех возможных путей между заданными узлами. Это демонстрирует многошаговый поиск, который является ключевой частью PathRAG.

3. **Синтез контекста и генерация**: Мы используем LLM для генерации ответа на основе найденных путей. В реальной системе контекст будет более сложным, и генерация будет учитывать множество факторов, включая релевантность и полноту информации.

Этот пример демонстрирует, как PathRAG может использовать граф знаний для улучшения процесса извлечения и генерации ответов на сложные запросы.